# Task 61. Text processing

The pythonspot NLTK link in the outline no longer works (404). This notebook covers the
same NLTK topics that category had: tokenizing, stop words, stemming, speech tagging,
a names-based gender classifier, and sentiment analysis.

In [1]:
# ===== Setup: download the NLTK data used in this notebook =====
import nltk

for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
                 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
                 'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words', 'names', 'movie_reviews',
                 'tagsets', 'tagsets_json']:
    nltk.download(resource, quiet=True)

In [2]:
# ===== NLTK: tokenize text =====
from nltk.tokenize import sent_tokenize, word_tokenize

data = "All work and no play makes jack a dull boy. All work and no play makes jack a dull boy."
print(word_tokenize(data))
print(sent_tokenize(data))

['All', 'work', 'and', 'no', 'play', 'makes', 'jack', 'a', 'dull', 'boy', '.', 'All', 'work', 'and', 'no', 'play', 'makes', 'jack', 'a', 'dull', 'boy', '.']
['All work and no play makes jack a dull boy.', 'All work and no play makes jack a dull boy.']


In [3]:
# store the tokens of each sentence in a list
sentences = sent_tokenize("Mr. Smith arrived at 9 a.m. today. He was late! Was the train delayed?")
tokens_per_sentence = [word_tokenize(s) for s in sentences]
tokens_per_sentence

[['Mr.', 'Smith', 'arrived', 'at', '9', 'a.m.', 'today', '.'],
 ['He', 'was', 'late', '!'],
 ['Was', 'the', 'train', 'delayed', '?']]

In [4]:
# ===== NLTK: stop words =====
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
data = "All work and no play makes jack dull boy. All work and no play makes jack a dull boy."
words = word_tokenize(data)

words_filtered = [w for w in words if w.lower() not in stop_words]
print(words_filtered)

['work', 'play', 'makes', 'jack', 'dull', 'boy', '.', 'work', 'play', 'makes', 'jack', 'dull', 'boy', '.']


In [5]:
# ===== NLTK: stemming =====
from nltk.stem import PorterStemmer

ps = PorterStemmer()
for w in ["game", "gaming", "gamed", "games"]:
    print(w, "->", ps.stem(w))

game -> game
gaming -> game
gamed -> game
games -> game


In [6]:
sentence = "gaming, the gamers play games"
for w in word_tokenize(sentence):
    print(w, ":", ps.stem(w))

gaming : game
, : ,
the : the
gamers : gamer
play : play
games : game


In [7]:
# ===== NLTK: speech (part-of-speech) tagging =====
from nltk import pos_tag

document = "Whether you're new to programming or an experienced developer, it's easy to learn and use Python."
for sent in sent_tokenize(document):
    print(pos_tag(word_tokenize(sent)))

[('Whether', 'IN'), ('you', 'PRP'), ("'re", 'VBP'), ('new', 'JJ'), ('to', 'TO'), ('programming', 'VBG'), ('or', 'CC'), ('an', 'DT'), ('experienced', 'JJ'), ('developer', 'NN'), (',', ','), ('it', 'PRP'), ("'s", 'VBZ'), ('easy', 'JJ'), ('to', 'TO'), ('learn', 'VB'), ('and', 'CC'), ('use', 'VB'), ('Python', 'NNP'), ('.', '.')]


In [8]:
# keep only the nouns (tags starting with NN)
tagged = pos_tag(word_tokenize(document))
nouns = [word for word, tag in tagged if tag.startswith('NN')]
nouns

['developer', 'Python']

In [9]:
# ===== Natural Language Processing prediction: gender from a name =====
import random
from nltk.corpus import names
from nltk import NaiveBayesClassifier, classify

def gender_features(word):
    return {'last_letter': word[-1].lower(), 'last_two': word[-2:].lower()}

labeled_names = ([(n, 'male') for n in names.words('male.txt')] +
                 [(n, 'female') for n in names.words('female.txt')])
random.seed(1)
random.shuffle(labeled_names)

featuresets = [(gender_features(n), g) for n, g in labeled_names]
train_set, test_set = featuresets[500:], featuresets[:500]

classifier = NaiveBayesClassifier.train(train_set)
print("Accuracy:", classify.accuracy(classifier, test_set))

Accuracy:

 0.798


In [10]:
for name in ['Frank', 'Emma', 'Ahmed', 'Sara', 'Oliver', 'Mia']:
    print(name, '->', classifier.classify(gender_features(name)))

Frank -> male
Emma -> female
Ahmed -> male
Sara -> female
Oliver -> male
Mia -> female


In [11]:
classifier.show_most_informative_features(8)

Most Informative Features
                last_two = 'na'           female : male   =    166.2 : 1.0
                last_two = 'la'           female : male   =     77.8 : 1.0
                last_two = 'ia'           female : male   =     39.7 : 1.0
             last_letter = 'a'            female : male   =     35.7 : 1.0
                last_two = 'sa'           female : male   =     35.6 : 1.0
             last_letter = 'k'              male : female =     32.1 : 1.0
                last_two = 'us'             male : female =     29.6 : 1.0
                last_two = 'ra'           female : male   =     27.0 : 1.0


In [12]:
# ===== Sentiment analysis with NLTK (movie reviews) =====
from nltk.corpus import movie_reviews

documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]
random.shuffle(documents)

# use the 2000 most frequent words as features
all_words = nltk.FreqDist(w.lower() for w in movie_reviews.words())
word_features = [w for w, _ in all_words.most_common(2000)]

def document_features(document):
    words = set(w.lower() for w in document)
    return {f'contains({w})': (w in words) for w in word_features}

featuresets = [(document_features(d), c) for d, c in documents]
train_set, test_set = featuresets[200:], featuresets[:200]

sentiment_classifier = NaiveBayesClassifier.train(train_set)
print("Accuracy:", classify.accuracy(sentiment_classifier, test_set))

Accuracy: 0.84


In [13]:
sentiment_classifier.show_most_informative_features(10)

Most Informative Features
   contains(outstanding) = True              pos : neg    =     10.3 : 1.0
   contains(wonderfully) = True              pos : neg    =      7.6 : 1.0
        contains(seagal) = True              neg : pos    =      7.0 : 1.0
         contains(damon) = True              pos : neg    =      6.4 : 1.0
         contains(mulan) = True              pos : neg    =      6.4 : 1.0
          contains(lame) = True              neg : pos    =      5.8 : 1.0
         contains(flynt) = True              pos : neg    =      5.7 : 1.0
        contains(wasted) = True              neg : pos    =      5.5 : 1.0
         contains(awful) = True              neg : pos    =      5.4 : 1.0
        contains(poorly) = True              neg : pos    =      5.0 : 1.0


In [14]:
for review in ["This movie was wonderful, the acting was outstanding and the story was memorable",
               "A boring, stupid film with terrible acting and a ridiculous plot"]:
    print(sentiment_classifier.classify(document_features(word_tokenize(review))), ':', review)

neg : This movie was wonderful, the acting was outstanding and the story was memorable
neg : A boring, stupid film with terrible acting and a ridiculous plot
